# 31 — Logistic Regression baseline

Baseline lineal escalable mediante **SGD con pérdida logística**, one-hot acotado y entrenamiento streaming sobre TRAIN completo. La evaluación permanece en VALIDATION; TEST sigue sellado.

In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m3_gate = paths.docs / "decisions" / "M3_GATE.md"
assert m3_gate.exists() and "M3 ✅ CLOSED" in m3_gate.read_text(encoding="utf-8"), (
    "M3 no está formalmente cerrado. No iniciar Modeling."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MODEL_DIR = paths.root / "artifacts" / "models" / "m4"
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

{
    "gold_schema": GOLD_SPEC.schema_version,
    "modeling_schema": MODEL_SPEC.schema_version,
    "target": MODEL_SPEC.target,
    "model_matrix": MODEL_SPEC.model_matrix,
    "operational_review_fraction": MODEL_SPEC.ranking.operational_fraction,
}

assert (paths.reports / "modeling" / "30_validation.parquet").exists(), "Ejecuta 30 primero."
from argos.models.linear import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm4-31-streaming-logistic-v1'

In [2]:
from argos.models.linear import train_linear_model

result = train_linear_model(
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    reports_root=paths.reports,
    model_dir=MODEL_DIR,
    manifest_dir=MANIFEST_DIR,
)
result.metrics


,model,split,rows,positives,prevalence,pr_auc,roc_auc,log_loss,brier,ece
0,logistic_sgd,validation,950670,9180,0.009656,0.038582,0.764114,1.497357,0.105533,0.106978


In [3]:
result.ranking

,model,split,review_fraction,reviewed,captured_positives,precision_at_k,recall_at_k,lift_at_k,cumulative_gain
0,logistic_sgd,validation,0.001,951,56,0.058885,0.006100,6.098101,0.006100
1,logistic_sgd,validation,0.005,4754,273,0.057425,0.029739,5.946899,0.029739
2,logistic_sgd,validation,0.010,9507,547,0.057537,0.059586,5.958418,0.059586
3,logistic_sgd,validation,0.025,23767,1283,0.053982,0.139760,5.590355,0.139760
4,logistic_sgd,validation,0.050,47534,2616,0.055034,0.284967,5.699286,0.284967
5,logistic_sgd,validation,0.100,95067,4834,0.050848,0.526580,5.265795,0.526580


In [4]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,linear_train_population_streamed,training,True,True,"{""epochs"": 2, ""expected"": 5563202, ""processed_...",streaming logistic consumes every TRAIN row in...
1,linear_validation_population,evaluation,True,True,"{""distinct_contracts"": 950670, ""expected"": 950...",linear validation predictions cover the comple...
2,linear_probabilities_finite,quality,True,True,"{""max"": 1.0, ""min"": 0.0}",linear validation scores are finite probabilities
3,linear_test_isolation,test_isolation,True,True,"{""consumed_splits"": [""train"", ""validation""]}",notebook 31 does not consume TEST rows


In [5]:
blocking = result.validation.loc[result.validation["blocking"]]
print("31 ✅ PASS" if result.ready_for_next_notebook else "31 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Model:", result.model_path)
assert result.ready_for_next_notebook


31 ✅ PASS
Blocking checks: 4/4
Model: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/models/m4/logistic_sgd.joblib
